<a href="https://colab.research.google.com/github/Rishhhi/async-vlm-extraction-api/blob/main/main_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Core inference engine and multimodal utilities
!pip install -q --upgrade transformers
!pip install -q torch pillow accelerate bitsandbytes qwen-vl-utils

# API serving and asynchronous execution environment
!pip install -q fastapi uvicorn pydantic nest-asyncio

# Frontend UI and secure network tunneling
!pip install -q gradio pyngrok

In [ ]:
# ==============================================================================
# Phase 2: Model Instantiation & Memory-Efficient Loading
# ==============================================================================
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig

# Define target model repository
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# --- 1. Quantization Configuration ---
# Enforce NormalFloat4 (NF4) quantization to fit the 2B model within the T4's 16GB VRAM.
# Double quantization is enabled to further reduce the memory footprint of quantization constants,
# while the compute dtype is kept at FP16 to maintain inference throughput.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# --- 2. Processor Initialization ---
print(f"Initializing processor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

# --- 3. Model Weight Loading & Device Mapping ---
# 'device_map="auto"' delegates memory management to Accelerate,
# ensuring optimal placement across available GPU/CPU resources.
print("Fetching weights and applying on-the-fly quantization...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config
)

print("Engine initialized: Qwen-VL loaded successfully.")

In [ ]:
# ==============================================================================
# Phase 3: Multimodal Inference & Data Extraction
# ==============================================================================
import requests
from io import BytesIO
from PIL import Image
from qwen_vl_utils import process_vision_info
import torch

# --- 1. Data Ingestion ---
# Fetching the source document. In a production environment, this would be
# replaced by a direct pull from an S3 bucket or a local temp directory.
DOCUMENT_URL = "https://templates.invoicehome.com/invoice-template-us-neat-750px.png"
response = requests.get(DOCUMENT_URL)
image = Image.open(BytesIO(response.content))
display(image.resize((350, 450)))

# --- 2. Payload Construction ---
# Define the multimodal chat template. Strict JSON formatting is enforced
# in the prompt to ensure the output is directly parsable by downstream APIs.
extraction_prompt = (
    "Extract the 'Invoice Number', 'Total Due', and 'Date'. "
    "Output strictly as a valid JSON object without any markdown formatting."
)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": extraction_prompt}
        ]
    }
]

# --- 3. Tensor Preprocessing ---
# Translate the raw image and text into the specific tensor formats required by Qwen.
# process_vision_info handles the complex patch-embedding math for the vision encoder.
text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text_input],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda") # Map tensors directly to GPU VRAM

# --- 4. Inference Execution ---
# torch.no_grad() is mandatory here. It disables gradient calculation,
# preventing severe VRAM leaks during forward passes in production.
print("Executing layout analysis and data extraction...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

# --- 5. Post-Processing & Decoding ---
# Isolate the newly generated tokens from the input prompt tokens,
# then decode the raw mathematical IDs back into a JSON string.
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_json = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

print("\n--- EXTRACTION PAYLOAD ---")
print(output_json)

In [ ]:
# ==============================================================================
# Phase 4: Asynchronous API Service & Background Execution
# ==============================================================================
import threading
import requests
from io import BytesIO
from PIL import Image

import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from qwen_vl_utils import process_vision_info

# Patch asyncio to enable Uvicorn to run within the Colab event loop
nest_asyncio.apply()

# --- 1. Service Initialization ---
app = FastAPI(
    title="Multimodal Document Extraction API",
    description="Asynchronous endpoint for layout-aware document extraction using Qwen-VL.",
    version="1.0.0"
)

# --- 2. Data Validation Schema ---
# Enforces strict payload structure to prevent malformed requests from reaching the GPU.
class ExtractionPayload(BaseModel):
    image_url: str
    prompt: str

# --- 3. Inference Endpoint ---
@app.post("/api/v1/extract")
async def extract_document_data(payload: ExtractionPayload):
    """Processes an image URL and a prompt to extract structured data via the VLM."""
    try:
        # Document Ingestion (with standard production timeouts)
        response = requests.get(payload.image_url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))

        # Prompt Construction
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": payload.prompt}
            ]
        }]

        # Tensor Formatting
        text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text_input],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        # Stateless GPU Inference
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=150)

        # Output Decoding
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        extracted_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

        return {"status": "success", "data": extracted_text}

    except requests.exceptions.RequestException as req_err:
        raise HTTPException(status_code=400, detail=f"Image download failed: {str(req_err)}")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")

# --- 4. Background Server Execution ---
# Spawns a daemon thread to run the ASGI server without blocking the Jupyter kernel.
def start_asgi_server():
    print("Starting FastAPI ASGI server on http://127.0.0.1:8000/api/v1/extract ...")
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_asgi_server, daemon=True)
server_thread.start()

In [ ]:
# ==============================================================================
# Phase 5: Client-Side Integration Testing
# ==============================================================================
import time
import json
import requests

def test_extraction_endpoint(api_url: str, image_url: str, prompt: str):
    """Simulates a frontend client dispatching a payload to the extraction API."""

    payload = {
        "image_url": image_url,
        "prompt": prompt
    }

    print(f"Initiating POST request to {api_url}...")
    start_time = time.time()

    try:
        # Dispatch network request with a defined timeout to prevent infinite hanging
        response = requests.post(api_url, json=payload, timeout=45)

        # Calculate round-trip latency
        latency = round(time.time() - start_time, 2)

        print(f"Status Code: {response.status_code} | Latency: {latency}s")
        print("\n--- API Payload Response ---")

        # Parse and display output
        if response.status_code == 200:
            print(json.dumps(response.json(), indent=4))
        else:
            print(f"Server Error Detail: {response.text}")

    except requests.exceptions.ConnectionError:
        print("Connection failed. Ensure the FastAPI ASGI server thread is running.")
    except requests.exceptions.Timeout:
        print("Request timed out. The GPU inference may be taking too long.")

# --- Execute Integration Test ---
TARGET_ENDPOINT = "http://127.0.0.1:8000/api/v1/extract"
TEST_IMAGE = "https://templates.invoicehome.com/invoice-template-us-neat-750px.png"
TEST_PROMPT = "Extract strictly the 'Invoice Number' and the 'Total Due'. Return only valid JSON."

test_extraction_endpoint(TARGET_ENDPOINT, TEST_IMAGE, TEST_PROMPT)

In [ ]:
# ==============================================================================
# Phase 6: External Network Exposure & Ingress Tunneling
# ==============================================================================
from pyngrok import ngrok
import os
def establish_secure_ingress(port: int, auth_token: str) -> str:
    """Establishes a secure reverse proxy to expose the local ASGI server to the public web."""

    if not auth_token or auth_token == "YOUR_AUTH_TOKEN":
        print("Configuration Error: A valid Ngrok authentication token is required.")
        return None

    print(f"Initializing secure ingress tunnel on localhost:{port}...")

    try:
        # Apply authentication credentials
        ngrok.set_auth_token(auth_token)

        # Open the tunnel connection
        tunnel = ngrok.connect(port)

        print("✅ Ingress Tunnel Established")
        # Appending the specific API route for immediate testing convenience
        print(f"🌐 Public API Gateway: {tunnel.public_url}/api/v1/extract")

        return tunnel.public_url

    except Exception as e:
        print(f"Tunnel initialization failed: {str(e)}")
        return None

# --- Execute Tunnel Configuration ---
# Note: In a production environment, secrets must be injected via environment
# variables (e.g., os.getenv('NGROK_TOKEN')) or a secure credential vault.
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")
ASGI_PORT = 8000

public_endpoint = establish_secure_ingress(ASGI_PORT, NGROK_AUTH_TOKEN)

In [ ]:
# ==============================================================================
# Phase 7: Decoupled Frontend Interface (Gradio)
# ==============================================================================
import json
import requests
import gradio as gr

# --- Configuration ---
# Targets the local ASGI server configured in Phase 4
BACKEND_API_URL = "http://127.0.0.1:8000/api/v1/extract"

# --- 1. API Gateway Integration ---
def invoke_extraction_service(image_url: str, prompt: str) -> str:
    """Acts as the middleware proxy, forwarding UI inputs to the ML backend."""
    payload = {
        "image_url": image_url,
        "prompt": prompt
    }

    try:
        # Enforce a strict timeout to prevent UI thread blocking
        response = requests.post(BACKEND_API_URL, json=payload, timeout=60)

        if response.status_code == 200:
            return json.dumps(response.json(), indent=4)
        return f"HTTP Error {response.status_code}: {response.text}"

    except requests.exceptions.RequestException as e:
        return f"Service Unreachable: Ensure the FastAPI backend is running.\nDetails: {str(e)}"

# --- 2. UI Component Tree ---
# Utilizing Blocks for a customized, enterprise-style layout
with gr.Blocks(theme=gr.themes.Soft(), title="VLM Extraction Interface") as application_ui:
    gr.Markdown("# 📄 Multimodal Document Extraction")
    gr.Markdown("Asynchronous frontend interface coupled to the locally hosted Qwen-VL backend.")

    with gr.Row():
        # Input Column
        with gr.Column(scale=1):
            ui_image_url = gr.Textbox(
                label="Target Document (URL)",
                placeholder="https://example.com/invoice.png",
                value="https://templates.invoicehome.com/invoice-template-us-neat-750px.png"
            )
            ui_prompt = gr.Textbox(
                label="Extraction Directive",
                value="Extract strictly the 'Invoice Number' and the 'Total Due'. Return only valid JSON.",
                lines=4
            )
            ui_submit_btn = gr.Button("Execute Extraction", variant="primary")

        # Output Column
        with gr.Column(scale=1):
            ui_output = gr.Code(label="Backend JSON Response", language="json", interactive=False)

    # Event Binding
    ui_submit_btn.click(
        fn=invoke_extraction_service,
        inputs=[ui_image_url, ui_prompt],
        outputs=ui_output
    )

# --- 3. Application Launch ---
print("Initializing Gradio UI server...")
# share=True establishes a public ingress proxy specifically for the UI
application_ui.launch(share=True, inline=False)